In [1]:
%cd /home/our_storage/liam_sbhoo/dl-lab-final-project

/mnt/disks/our_storage/liam_sbhoo/dl-lab-final-project


In [2]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
from omegaconf import OmegaConf
from lavis.models import load_preprocess
from models.localized_blip2_qformer import LocalizedBlip2QFormer
from datasets.coco_data_module import CocoCaptionDataModule
from datasets.refcoco_dataset import RefCocoDataset

2023-07-31 15:48:50.696431: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-07-31 15:48:50.751651: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-31 15:48:51.664752: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-07-31 15:48:53.055125: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at htt

In [3]:
model_name = "blip2"
model_type = "pretrain"
model = LocalizedBlip2QFormer.from_pretrained(model_type=model_type).to("cuda")
cfg = OmegaConf.load(LocalizedBlip2QFormer.default_config_path(model_type)).preprocess
vis_preprocess, text_preprocess = load_preprocess(cfg)

In [4]:
STORAGE_DIR = "/home/our_storage"
refcoco = RefCocoDataset(
    vis_root=os.path.join(STORAGE_DIR, "shared/datasets/coco2014/coco/images/"),
    instances_file=os.path.join(STORAGE_DIR, "shared/datasets/refcocog/instances.json"),
    refs_file=os.path.join(STORAGE_DIR, "shared/datasets/refcocog/refs(umd).p"),
    dataset_name="refcoco_g",
    set_type="val",
    vis_transform=vis_preprocess["eval"],
    text_transform=text_preprocess["eval"],
    feed_image_mode="full"
)
dataloader = DataLoader(refcoco, batch_size=32, num_workers=4, collate_fn=RefCocoDataset.collate_fn)


Missing 0/2573 references for val set.


In [5]:
with torch.cuda.amp.autocast():
    score_matrix_i2t, score_matrix_t21 = model.compute_sim_matrix(dataloader, topk=1)

Computing features for evaluation...
batch  0
batch  1
batch  2
batch  3
batch  4
batch  5
batch  6
batch  7
batch  8
batch  9
batch  10
batch  11
batch  12
batch  13
batch  14
batch  15
batch  16
batch  17
batch  18
batch  19
batch  20
batch  21
batch  22
batch  23
batch  24
batch  25
batch  26
batch  27
batch  28
batch  29
batch  30
batch  31
batch  32
batch  33
batch  34
batch  35
batch  36
batch  37
batch  38
batch  39
batch  40
batch  41
batch  42
batch  43
batch  44
batch  45
batch  46
batch  47
batch  48
batch  49
batch  50
batch  51
batch  52
batch  53
batch  54
batch  55
batch  56
batch  57
batch  58
batch  59
batch  60
batch  61
batch  62
batch  63
batch  64
batch  65
batch  66
batch  67
batch  68
batch  69
batch  70
batch  71
batch  72
batch  73
batch  74
batch  75
batch  76
batch  77
batch  78
batch  79
batch  80
Done generating embeddings


Evaluation: Computing similarity matrix: 100%|██████████| 2573/2573 [01:05<00:00, 39.00it/s]


In [6]:
score_matrix_i2t.shape

(2573, 2573)

In [7]:
score_matrix_t21.shape

(2573, 2573)